# Notebook 02 — Preprocessing
## STOP 4 — Split with Stratification

In [1]:
from pathlib import Path
import json
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import TensorDataset, DataLoader

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_PATH = ROOT / "data" / "raw" / "winequality-red.csv"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, sep=";")
df["quality_class"] = df["quality"].apply(lambda q: 0 if q <= 4 else (1 if q <= 6 else 2))

X = df.drop(columns=["quality", "quality_class"])
y = df["quality_class"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42, stratify=y_train)

print("train counts:\n", y_train.value_counts().sort_index())
print("val counts:\n", y_val.value_counts().sort_index())
print("test counts:\n", y_test.value_counts().sort_index())

train counts:
 quality_class
0     42
1    897
2    148
Name: count, dtype: int64
val counts:
 quality_class
0      8
1    158
2     26
Name: count, dtype: int64
test counts:
 quality_class
0     13
1    264
2     43
Name: count, dtype: int64


## STOP 5 — Scale & Encode

In [2]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

with open(PROC / "scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

feature_names = X.columns.tolist()
(PROC / "feature_names.json").write_text(json.dumps(feature_names, indent=2))

np.savez(PROC / "dataset_arrays.npz",
         X_train=X_train_s.astype(np.float32),
         X_val=X_val_s.astype(np.float32),
         X_test=X_test_s.astype(np.float32),
         y_train=y_train.to_numpy(dtype=np.int64),
         y_val=y_val.to_numpy(dtype=np.int64),
         y_test=y_test.to_numpy(dtype=np.int64))

print("saved scaler and arrays")

saved scaler and arrays


## STOP 6 — DataLoader Setup

In [3]:
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
X_val_t = torch.tensor(X_val_s, dtype=torch.float32)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train.to_numpy(), dtype=torch.long)
y_val_t = torch.tensor(y_val.to_numpy(), dtype=torch.long)
y_test_t = torch.tensor(y_test.to_numpy(), dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=64, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=64, shuffle=False)

print(len(train_loader), len(val_loader), len(test_loader))

17 3 5
